In [0]:
alphacollector_transactions_history=dbutils.widgets.get("alphacollector_transactions_history")
cash_invoice=dbutils.widgets.get("cash_invoice")
office=dbutils.widgets.get("office")
alphacollector_claims_history=dbutils.widgets.get("alphacollector_claims_history")
client=dbutils.widgets.get("client")
payerdimension=dbutils.widgets.get("payerdimension")
paymentstype=dbutils.widgets.get("paymenttype")
paymentsdetails=dbutils.widgets.get("paymentsdetails")

In [0]:
spark.sql(
    f"""
CREATE OR REPLACE TEMP VIEW pyt_temp AS
SELECT 
    EntryDate, 
    TransDate, 
    TransCode, 
    TransType, 
    PaymentID, 
    BatchID, 
    OfficeExternalId, 
    ClaimNumber, 
    SUM(TransAmt) AS CollectedCash, 
    ReportingWeekEndingDate
FROM {alphacollector_transactions_history}
WHERE TransType = 'Payment' 
    AND PaymentId <> ' ' 
    AND ReportingWeekEndingDate = DATE(DATE_ADD(CURRENT_DATE(), -4))
GROUP BY 
    TransCode, 
    TransType, 
    PaymentID, 
    BatchID, 
    OfficeExternalId, 
    ClaimNumber, 
    ReportingWeekEndingDate,
    EntryDate, 
    TransDate;
"""
)

In [0]:
spark.sql(
    f"""
INSERT INTO {cash_invoice}
(
    source_system_key,
    reporting_week_ending_date_key,
    posted_date_key,
    payor_key,
    client_key,
    cash_collected,
    deposit_date_key,
    batch_id,
    check_id,
    office_key,
    type,
    bank,
    agency_id,
    payor_category,
    invoice_number,
    rap_payment,
    final_payment,
    other_payment,
    refund_payment,
    unapplied_cash,
    deposit_number,
    hchb_date_entered_key,
    month_end_close_reporting_period
)
SELECT 
    19 AS source_system_key,
    TRY_CAST(REPLACE(CAST(pyt.ReportingWeekEndingDate AS STRING), '-', '') AS INT) AS reporting_week_ending_date_key,
    TRY_CAST(
        CASE 
            WHEN pyt.EntryDate IS NULL OR TRIM(pyt.EntryDate) = '' THEN NULL
            WHEN INSTR(pyt.EntryDate, '/') = 0 THEN NULL
            ELSE CONCAT(
                YEAR(TO_DATE(pyt.EntryDate, 'M/d/yyyy')),
                '0',
                LPAD(MONTH(TO_DATE(pyt.EntryDate, 'M/d/yyyy')), 2, '0'),
                SUBSTRING(pyt.EntryDate, 4, 2)
            )
        END 
    AS INT) AS posted_date_key,
    f.PayerKey AS payor_key,
    d.ClientKey AS client_key,
    CAST(pyt.CollectedCash AS FLOAT) AS cash_collected,
    TRY_CAST(
        CASE 
            WHEN pyt.TransDate IS NULL OR TRIM(pyt.TransDate) = '' THEN NULL
            WHEN INSTR(pyt.TransDate, '/') = 0 THEN NULL
            ELSE CONCAT(
                YEAR(TO_DATE(pyt.TransDate, 'M/d/yyyy')),
                '0',
                LPAD(MONTH(TO_DATE(pyt.TransDate, 'M/d/yyyy')), 2, '0'),
                SUBSTRING(pyt.TransDate, 4, 2)
            )
        END 
    AS INT) AS deposit_date_key,
    pyt.BatchId AS batch_id,
    pyt.PaymentId AS check_id,
    ofc.OfficeKey AS office_key,
    pyt.TransCode AS type,
    NULL AS bank,
    TRY_CAST(pyt.OfficeExternalId AS INT) AS agency_id,
    CAST(f.TypeID AS STRING) AS payor_category,
    pyt.ClaimNumber AS invoice_number,
    NULL AS rap_payment,
    NULL AS final_payment,
    NULL AS other_payment,
    NULL AS refund_payment,
    NULL AS unapplied_cash,
    NULL AS deposit_number,
    NULL AS hchb_date_entered_key,
    NULL AS month_end_close_reporting_period
FROM pyt_temp pyt
LEFT JOIN {office} ofc
    ON ofc.OfficeNumber = pyt.OfficeExternalId
LEFT JOIN (
    SELECT * FROM (
        SELECT 
            ClaimNumber, 
            MedicalRecordNumber, 
            OfficeExternalId, 
            PayerName,
            ROW_NUMBER() OVER (PARTITION BY ClaimNumber, OfficeExternalId ORDER BY LoadDate DESC) AS rnb
        FROM {alphacollector_claims_history}
    ) a
    WHERE rnb = 1
) b
    ON b.ClaimNumber = pyt.ClaimNumber 
    AND b.OfficeExternalId = pyt.OfficeExternalId
LEFT JOIN (
    SELECT * FROM (
        SELECT 
            ClientKey, 
            MedicalRecordNumber, 
            ROW_NUMBER() OVER (PARTITION BY MedicalRecordNumber ORDER BY ClientKey DESC) AS rnb
        FROM {client}
        WHERE SourceSystem = 'CUBHUB'
    ) c
    WHERE rnb = 1
) d
    ON b.MedicalRecordNumber = d.MedicalRecordNumber
LEFT JOIN (
    SELECT 
        PayerKey, 
        Name, 
        TypeID 
    FROM (
        SELECT 
            *, 
            ROW_NUMBER() OVER (PARTITION BY Name ORDER BY PayerKey DESC) AS rnb
        FROM {payerdimension}
        WHERE SourceSystemKey = 19
    ) e
    WHERE rnb = 1
) f
    ON f.Name = b.PayerName
LEFT JOIN (
    SELECT * 
    FROM {paymentstype}
    WHERE SourceSystem = 'CUBHUB'
) g
    ON g.PaymentTypeDescription = pyt.TransCode
LEFT JOIN (
    SELECT * 
    FROM {paymentsdetails}
    WHERE SourceSystem = 'CUBHUB'
) h
    ON COALESCE(h.batchid, '') = COALESCE(pyt.BatchID, '')
    AND COALESCE(TRY_CAST(h.AgencyID AS STRING), '') = COALESCE(TRY_CAST(pyt.OfficeExternalID AS STRING), '')
    AND COALESCE(h.checkID, '') = COALESCE(pyt.PaymentID, '')
    AND COALESCE(h.batchnumber, '') = COALESCE(pyt.BatchId, '')
    AND COALESCE(h.DepositID, '') = COALESCE(pyt.BatchId, '')
    AND h.SourceSystem = 'CUBHUB';
"""
)